# Fuse Lidar + Camera: GraphDECO 3DGS Smoke Test

Goal: prove that the exported `images/` + `sparse/0/` dataset can be opened by the official GraphDECO Gaussian Splatting trainer and reach early optimization iterations.

This is a loader/training smoke test, not a quality benchmark. Our current sparse points are lidar/camera diagnostic seeds, not true COLMAP feature tracks, so a weak-looking splat is expected.

In [ ]:
# 1. Confirm the notebook runtime has a CUDA GPU.
# In Colab: Runtime -> Change runtime type -> GPU.
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

Upload the ZIP produced locally by `reconstruction/package_graphdeco_dataset.py`. For the current test it should look like:

`20260625T214456Z-steady-undistorted-graphdeco.zip`

In [ ]:
# 2. Upload the packaged dataset ZIP from your workstation.
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No ZIP uploaded')
ZIP_PATH = Path('/content') / next(iter(uploaded.keys()))
print('Dataset zip:', ZIP_PATH)

In [ ]:
# 3. Clone and install the official GraphDECO implementation.
# This compiles CUDA extensions, so failures here usually mean the runtime/PyTorch/CUDA toolchain do not match.
%cd /content
!rm -rf gaussian-splatting
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd /content/gaussian-splatting
!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q plyfile tqdm opencv-python joblib ninja

# GraphDECO CUDA submodules import torch from setup.py. In Colab, pip's default
# build isolation can hide torch and fail with "Getting requirements to build wheel".
# --no-build-isolation makes the build use this runtime's installed torch/CUDA.
!python - <<'PY'
import torch
print('torch', torch.__version__)
print('torch cuda', torch.version.cuda)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device', torch.cuda.get_device_name(0))
PY
!nvcc --version
!python -m pip install -v --no-build-isolation ./submodules/diff-gaussian-rasterization
!python -m pip install -v --no-build-isolation ./submodules/simple-knn
# fused-ssim is optional; training falls back to Python SSIM if this fails.
!python -m pip install -v --no-build-isolation ./submodules/fused-ssim || true

In [ ]:
# 4. Unpack and sanity-check the dataset folder shape GraphDECO expects.
import json
import shutil
import zipfile
from pathlib import Path

dataset_root = Path('/content/fuse_graphdeco_dataset') / ZIP_PATH.stem
shutil.rmtree(dataset_root, ignore_errors=True)
dataset_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(dataset_root)

required = [
    dataset_root / 'images',
    dataset_root / 'sparse' / '0' / 'cameras.bin',
    dataset_root / 'sparse' / '0' / 'images.bin',
    dataset_root / 'sparse' / '0' / 'points3D.bin',
    dataset_root / 'sparse' / '0' / 'points3D.ply',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))

images = sorted((dataset_root / 'images').glob('*'))
print('dataset_root:', dataset_root)
print('image count:', len(images))
print('sparse files:', sorted(path.name for path in (dataset_root / 'sparse' / '0').glob('*')))

report_path = dataset_root / 'graphdeco_input_check.json'
if report_path.exists():
    print(json.dumps(json.loads(report_path.read_text()), indent=2)[:3000])

In [ ]:
# 5. Run a tiny training smoke test.
# Success criterion: train.py starts, loads cameras/images/points, and reaches the requested iteration.
import shlex
import subprocess
from pathlib import Path

out_dir = Path('/content/fuse_3dgs_output') / f'{dataset_root.name}-smoke'
out_dir.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    'python', 'train.py',
    '-s', str(dataset_root),
    '-m', str(out_dir),
    '--images', 'images',
    '--resolution', '8',
    '--iterations', '300',
    '--test_iterations', '300',
    '--save_iterations', '300',
    '--data_device', 'cpu',
    '--disable_viewer',
]
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, cwd='/content/gaussian-splatting', check=True)
print('output:', out_dir)
print('output files:', sorted(path.name for path in out_dir.glob('*')))

In [ ]:
# 6. Optional: inspect outputs and download them.
from google.colab import files
from pathlib import Path
import shutil

for path in sorted(out_dir.rglob('*'))[:80]:
    print(path.relative_to(out_dir))

archive_base = shutil.make_archive(str(out_dir), 'zip', out_dir)
print('download archive:', archive_base)
files.download(archive_base)